# Qwen3.6-27B Dense: Multi-Layer SAE for Capability Steering

First SAE ensemble on Qwen3.6-27B dense (released 2026-04-21). 3 layers × TopK SAE → find features correlating with correctness → steer at inference.

## Pipeline

1. Setup + load Qwen3.6-27B
2. Install hooks at **L11, L31, L55** (early/mid/late, all GA layers)
3. Generate 150 Stage B rollouts with per-token residual capture
4. Upload activations to HF (crash-safety)
5. Train 3 TopK SAEs (n=4096, k=32, 20 epochs)
6. Feature discovery: correlate each feature with correctness
7. Semantic characterization: top activating tokens per feature
8. Steering eval: baseline + 3 single-layer + combined multi-layer
9. Upload results

## Budget

~14h total on RTX 6000 Blackwell. Crash-safe at each stage.

## Decision rules

- **Any single layer +3pp**: single-layer steering works, scale that direction
- **Combined +5pp but single <+1pp**: multi-layer compound effect, novel paper angle
- **Tudo <+1pp**: replica 35B paper null in dense — update memory, pivot

## Cell 1 — Setup

In [ ]:
import sys, subprocess, os, shutil
from pathlib import Path
def pip(*a): return subprocess.run([sys.executable, '-m', 'pip', *a], check=False)

try:
    import transformers
    from transformers.models.auto.configuration_auto import CONFIG_MAPPING_NAMES
    ok = 'qwen3_5' in CONFIG_MAPPING_NAMES
except Exception: ok = False

if not ok:
    pip('install', '-q', 'accelerate', 'datasets', 'huggingface_hub==1.5.0',
        'safetensors', 'einops', 'scikit-learn', 'sentencepiece', 'tokenizers',
        'protobuf', 'scipy', 'hf_transfer')
    pip('uninstall', '-y', 'transformers', 'causal-conv1d')
    SRC = '/content/transformers_src'
    if Path(SRC).exists(): shutil.rmtree(SRC)
    subprocess.run(['git','clone','--quiet','--depth=1',
                    'https://github.com/huggingface/transformers.git', SRC], check=True)
    pip('install','--force-reinstall','--no-deps','--no-cache-dir', SRC)
    pip('install','--no-cache-dir','flash-linear-attention')
    for m in list(sys.modules):
        if m.startswith('transformers') or m.startswith('huggingface_hub'): del sys.modules[m]

try:
    from google.colab import userdata
    t = userdata.get('HF_TOKEN')
    if t:
        from huggingface_hub import login
        login(token=t); print('HF auth OK')
except: pass

import json, re, time, random, pickle
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm
from collections import defaultdict, Counter
from safetensors import safe_open
from safetensors.numpy import save_file, load_file
from huggingface_hub import snapshot_download, HfApi, create_repo
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

OUT = Path('/content/sae27b'); OUT.mkdir(exist_ok=True)
MODEL_ID = 'Qwen/Qwen3.6-27B'
HF_STAGE_B = 'caiovicentino1/Qwen3.6-35B-A3B-mcr-stage-b'
HF_TARGET = 'caiovicentino1/qwen36-27b-sae-multilayer'

LAYERS = [11, 31, 55]  # early / mid / late GA layers
D_MODEL = 5120
N_FEATURES = 4096
TOPK = 32
SAE_EPOCHS = 20
SAE_BS = 4096
SAE_LR = 3e-4
N_ROLLOUTS = 150
MAX_NEW = 2500
MAX_TOKENS_PER_LAYER = 200_000  # cap for SAE train
FORCE_SUFFIX = '\n</think>\n\nFinal answer: \\boxed{'

print('setup done: 27B multi-layer SAE L11/L31/L55')

## Cell 2 — Load Qwen3.6-27B + install 3 hooks

In [ ]:
from transformers import AutoTokenizer, AutoModelForImageTextToText

tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tok.pad_token_id is None: tok.pad_token = tok.eos_token

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, device_map='cuda',
    attn_implementation='sdpa', trust_remote_code=True)
model.eval()
for p in model.parameters(): p.requires_grad_(False)
print(f'Qwen3.6-27B loaded: {torch.cuda.memory_allocated()/1e9:.1f} GB')

def get_layer_module(m, idx):
    cands = [m]
    if hasattr(m, 'model'): cands.append(m.model)
    for start in cands:
        for path in [('model','language_model','layers'),('language_model','layers'),('model','layers')]:
            cur = start; ok = True
            for p in path:
                if hasattr(cur, p): cur = getattr(cur, p)
                else: ok = False; break
            if ok and hasattr(cur, '__getitem__'):
                try: return cur[idx]
                except: continue
    raise RuntimeError(f'layer {idx} not found')

# Hooks capture LAST token at each forward (during generation, this is each new token)
captured = {L: [] for L in LAYERS}
def make_hook(L):
    def h(mod, inp, out):
        x = out[0] if isinstance(out, tuple) else out
        captured[L].append(x[:, -1, :].detach().float().cpu())
    return h

handles = []
for L in LAYERS:
    layer = get_layer_module(model, L)
    handles.append(layer.register_forward_hook(make_hook(L)))
print(f'OK {len(handles)} hooks on layers {LAYERS}')

## Cell 3 — Generate 150 rollouts with per-token residual capture

In [ ]:
corpus = snapshot_download(HF_STAGE_B, repo_type='dataset', cache_dir='/content/cache')
shards = sorted((Path(corpus)/'shards').glob('q*.safetensors'))

questions = []
seen = set()
for shard in shards:
    with safe_open(str(shard), framework='pt') as f:
        meta = f.metadata()
        q = meta['question']
        if q in seen: continue
        seen.add(q)
        opts = json.loads(meta['options'])
        if len(opts) != 10: continue
        questions.append(dict(question=q, options=opts, gold_letter=meta['gold_letter']))

random.seed(100)  # NEW seed — disjoint from original 27B probe training (seed 42)
random.shuffle(questions)
train_questions = questions[:N_ROLLOUTS]
print(f'{len(train_questions)} rollout questions (seed 100)')

def extract_answer(text):
    post = text.split('</think>')[-1] if '</think>' in text else text
    for pattern in [r'\\boxed\{([A-J])\}',
                    r'[Aa]nswer\s*(?:is\s*)?[:=]?\s*\*?\*?\(?([A-J])\)?\*?\*?']:
        m = re.search(pattern, post, re.MULTILINE)
        if m: return m.group(1).upper()
    m = re.findall(r'\\boxed\{([A-J])\}', text)
    if m: return m[-1]
    return None

def fmt(q, opts):
    choices = '\n'.join(f'{chr(65+i)}. {o}' for i, o in enumerate(opts))
    content = (f'Answer the following multiple-choice question. '
               f'Reason step by step, then put the letter in \\boxed{{}}.\n\n'
               f'Question: {q}\n\nOptions:\n{choices}')
    return tok.apply_chat_template([{'role':'user','content':content}],
                                    tokenize=False, add_generation_prompt=True, enable_thinking=True)

def force_close(full_ids, prompt_len):
    gen = tok.decode(full_ids[prompt_len:].tolist(), skip_special_tokens=False)
    if '</think>' in gen:
        return tok.decode(full_ids[prompt_len:].tolist(), skip_special_tokens=True)
    full = tok.decode(full_ids.tolist(), skip_special_tokens=False) + FORCE_SUFFIX
    ctx = tok(full, return_tensors='pt', add_special_tokens=False).input_ids.cuda()
    with torch.no_grad():
        out = model.generate(ctx, max_new_tokens=15, do_sample=False,
                             pad_token_id=tok.pad_token_id, use_cache=True)
    suf = tok.decode(out[0, ctx.shape[1]:].tolist(), skip_special_tokens=True)
    return tok.decode(full_ids[prompt_len:].tolist(), skip_special_tokens=True) + FORCE_SUFFIX + suf

# Storage: accumulate per-rollout per-layer activations (only RESPONSE tokens)
rollouts_data = []  # list of dicts: {question, correct, pred, L11_acts [T,d], L31_acts, L55_acts}
t0 = time.time()

for qi, q in enumerate(tqdm(train_questions, desc='gen rollouts')):
    p = fmt(q['question'], q['options'])
    ids = tok(p, return_tensors='pt').input_ids.cuda()
    if ids.shape[1] > 1200: continue  # keep context manageable
    for L in LAYERS: captured[L] = []

    try:
        with torch.no_grad():
            out = model.generate(ids, max_new_tokens=MAX_NEW, do_sample=False,
                                 pad_token_id=tok.pad_token_id, use_cache=True)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache(); continue

    text = force_close(out[0], ids.shape[1])
    pred = extract_answer(text)
    correct = (pred == q['gold_letter']) if pred else False

    # chunks[0] is prompt forward (captures last token), [1:] are per-generated-token
    for L in LAYERS:
        if len(captured[L]) < 2: break
    else:
        per_token_acts = {L: torch.stack(captured[L][1:], dim=0)[:, 0, :].float().cpu().numpy().astype(np.float16)
                          for L in LAYERS}  # [T, d]
        rollouts_data.append({
            'qi': qi, 'correct': correct, 'pred': pred, 'gold': q['gold_letter'],
            'acts': per_token_acts,
        })

    # Crash-safe checkpoint every 10
    if (qi+1) % 10 == 0:
        nc = sum(1 for r in rollouts_data if r['correct'])
        print(f'  [{qi+1}/{len(train_questions)}] kept {len(rollouts_data)}, {nc} correct ({nc/max(1,len(rollouts_data))*100:.0f}%)')
        # Save lightweight metadata (acts too big to checkpoint easily)
        with open(OUT / 'rollouts_meta.json', 'w') as f:
            json.dump([{'qi': r['qi'], 'correct': r['correct'], 'pred': r['pred'],
                         'gold': r['gold'], 'n_tokens': r['acts'][LAYERS[0]].shape[0]}
                        for r in rollouts_data], f, indent=2)

print(f'\n✅ {len(rollouts_data)} rollouts in {(time.time()-t0)/60:.1f} min')
nc = sum(1 for r in rollouts_data if r['correct'])
print(f'Correctness rate: {nc}/{len(rollouts_data)} = {nc/len(rollouts_data)*100:.1f}%')

# Total tokens per layer
total_tokens = sum(r['acts'][LAYERS[0]].shape[0] for r in rollouts_data)
print(f'Total response tokens per layer: {total_tokens:,}')

## Cell 4 — 🚨 Save activations to disk + upload to HF (crash-safety)

In [ ]:
# Stack per layer, cap at MAX_TOKENS_PER_LAYER
per_layer_acts = {}
per_token_labels = []  # correctness per token (inherited from rollout)

for L in LAYERS:
    stack = []
    labels = []
    for r in rollouts_data:
        acts = r['acts'][L]
        stack.append(acts)
        labels.extend([r['correct']] * acts.shape[0])
    X = np.concatenate(stack, axis=0)
    if X.shape[0] > MAX_TOKENS_PER_LAYER:
        idx = np.random.default_rng(42).permutation(X.shape[0])[:MAX_TOKENS_PER_LAYER]
        X = X[idx]
        labels = [labels[i] for i in idx]
    per_layer_acts[L] = X
    if L == LAYERS[0]: per_token_labels = np.array(labels, dtype=bool)
    print(f'L{L}: {X.shape} fp16 ({X.nbytes/1e6:.0f} MB)')

# Save + upload
save_file({f'L{L}': per_layer_acts[L] for L in LAYERS},
          str(OUT / 'activations.safetensors'))
np.save(OUT / 'labels.npy', per_token_labels)
print(f'saved activations.safetensors: {(OUT/"activations.safetensors").stat().st_size/1e6:.0f} MB')

api = HfApi()
create_repo(HF_TARGET, repo_type='model', private=False, exist_ok=True)
api.upload_file(path_or_fileobj=str(OUT/'activations.safetensors'),
                path_in_repo='activations.safetensors',
                repo_id=HF_TARGET, repo_type='model',
                commit_message='27B per-token activations L11/L31/L55')
api.upload_file(path_or_fileobj=str(OUT/'labels.npy'),
                path_in_repo='labels.npy',
                repo_id=HF_TARGET, repo_type='model',
                commit_message='per-token correctness labels')
print(f'✅ Uploaded: https://huggingface.co/{HF_TARGET}')

## Cell 5 — Train 3 TopK SAEs

In [ ]:
class TopKSAE(nn.Module):
    def __init__(self, d_in, n, k):
        super().__init__()
        self.W_enc = nn.Parameter(torch.empty(d_in, n))
        self.b_enc = nn.Parameter(torch.zeros(n))
        self.W_dec = nn.Parameter(torch.empty(n, d_in))
        self.b_dec = nn.Parameter(torch.zeros(d_in))
        self.k = k
        w = torch.randn(d_in, n) / (d_in ** 0.5)
        self.W_enc.data = w
        self.W_dec.data = w.T.contiguous() / w.T.norm(dim=-1, keepdim=True).clamp_min(1e-8)
    def encode(self, x):
        pre = (x - self.b_dec) @ self.W_enc + self.b_enc
        top_v, top_i = pre.topk(self.k, dim=-1)
        z = torch.zeros_like(pre)
        z.scatter_(-1, top_i, F.relu(top_v))
        return z, top_i, top_v
    def decode(self, z): return z @ self.W_dec + self.b_dec
    def forward(self, x):
        z, _, _ = self.encode(x); return self.decode(z), z

def train_sae(X_np, d_in, n, k, epochs, bs, lr, name=''):
    X = torch.from_numpy(X_np).float().cuda()
    sae = TopKSAE(d_in, n, k).cuda()
    opt = torch.optim.Adam(sae.parameters(), lr=lr)
    N = X.shape[0]; last_fire = torch.zeros(n, device='cuda'); step = 0
    for e in range(epochs):
        perm = torch.randperm(N, device='cuda')
        el = 0.0; nb = 0
        for i in range(0, N, bs):
            xb = X[perm[i:i+bs]]
            xh, z = sae(xb)
            loss = F.mse_loss(xh, xb)
            opt.zero_grad(); loss.backward(); opt.step()
            with torch.no_grad():
                sae.W_dec.data /= sae.W_dec.data.norm(dim=-1, keepdim=True).clamp_min(1e-8)
                fired = (z > 0).any(dim=0); last_fire[fired] = step
            step += 1; el += loss.item(); nb += 1
        if (e+1) % 5 == 0 or e == 0:
            dead = (step - last_fire) > 500
            n_dead = int(dead.sum().item())
            if n_dead > 0:
                with torch.no_grad():
                    nd = torch.randn(n_dead, d_in, device='cuda')
                    nd /= nd.norm(dim=-1, keepdim=True)
                    sae.W_dec.data[dead] = nd
                    sae.W_enc.data[:, dead] = nd.T
                    sae.b_enc.data[dead] = 0.0
            with torch.no_grad():
                xh, _ = sae(X[:10000])
                var_expl = 1 - ((xh - X[:10000]).var() / X[:10000].var()).item()
            print(f'  {name} ep{e+1:>2}/{epochs} mse={el/nb:.5f} var_expl={var_expl:.3f} dead_rev={n_dead}')
    return sae.cpu()

saes = {}
for L in LAYERS:
    print(f'\n=== Training SAE on L{L} (n={N_FEATURES}, k={TOPK}) ===')
    sae = train_sae(per_layer_acts[L], D_MODEL, N_FEATURES, TOPK,
                    SAE_EPOCHS, SAE_BS, SAE_LR, name=f'L{L}')
    saes[L] = sae
    torch.save(sae.state_dict(), OUT / f'sae_L{L}_n{N_FEATURES}_k{TOPK}.pt')
    torch.cuda.empty_cache()

# Upload SAEs
for L in LAYERS:
    api.upload_file(path_or_fileobj=str(OUT/f'sae_L{L}_n{N_FEATURES}_k{TOPK}.pt'),
                    path_in_repo=f'sae_L{L}_n{N_FEATURES}_k{TOPK}.pt',
                    repo_id=HF_TARGET, repo_type='model',
                    commit_message=f'TopK SAE L{L}')
print(f'\n✅ All 3 SAEs uploaded')

## Cell 6 — Feature discovery: correctness correlation per feature

In [ ]:
# For each layer's SAE, encode all per-token activations, compute per-feature correctness correlation
feature_stats = {}  # L -> list of (feat_id, auroc, fire_rate)
from sklearn.metrics import roc_auc_score

for L in LAYERS:
    sae = saes[L].cuda()
    X = torch.from_numpy(per_layer_acts[L]).float().cuda()
    N = X.shape[0]
    # Encode all tokens
    with torch.no_grad():
        z, _, _ = sae.encode(X)
    fires = (z > 0).cpu().numpy()  # [N, n_features]
    mean_fire_rate = fires.mean(axis=0)  # [n_features]

    labels = per_token_labels if L == LAYERS[0] else np.tile(per_token_labels, int(np.ceil(X.shape[0]/len(per_token_labels))))[:X.shape[0]]

    # Per-feature AUROC for predicting correctness from fire (binary)
    aurocs = []
    for fi in range(N_FEATURES):
        fr = fires[:, fi]
        if fr.sum() < 10 or fr.sum() > N - 10: aurocs.append(0.5); continue
        try:
            auc = roc_auc_score(labels, fr.astype(float))
            aurocs.append(auc)
        except: aurocs.append(0.5)
    aurocs = np.array(aurocs)

    # Top features: |AUROC - 0.5| largest (both directions — features that fire MORE on correct and on wrong)
    disc = np.abs(aurocs - 0.5)
    top_idx = np.argsort(-disc)[:30]
    feature_stats[L] = [(int(fi), float(aurocs[fi]), float(mean_fire_rate[fi])) for fi in top_idx]

    sae.cpu()
    print(f'\nL{L}: top-10 discriminative features')
    print(f'  {"feat":>5s} {"AUROC":>6s} {"fire%":>7s} {"direction":>10s}')
    for fi, auc, fr in feature_stats[L][:10]:
        direction = 'correct↑' if auc > 0.5 else 'wrong↑'
        print(f'  f{fi:>4d} {auc:>6.3f} {fr*100:>6.2f}% {direction:>10s}')

# Save feature stats
with open(OUT / 'feature_stats.json', 'w') as f:
    json.dump({f'L{L}': v for L, v in feature_stats.items()}, f, indent=2)
api.upload_file(path_or_fileobj=str(OUT/'feature_stats.json'),
                path_in_repo='feature_stats.json',
                repo_id=HF_TARGET, repo_type='model',
                commit_message='per-layer feature correctness AUROC')
print(f'\n✅ Feature stats uploaded')

## Cell 7 — Semantic characterization of top features

In [ ]:
# For top-5 features per layer, find activating tokens via forward on subset rollouts
TOP_FEATS_PER_LAYER = {L: [fi for fi, _, _ in feature_stats[L][:5]] for L in LAYERS}

# We need per-token TEXT, not just activation. Forward 10 rollouts with text tracking.
for h in handles: h.remove()
handles = []
for L in LAYERS:
    layer = get_layer_module(model, L)
    handles.append(layer.register_forward_hook(make_hook(L)))

random.seed(999)
char_questions = random.sample(questions, 10)
feat_contexts = {(L, fi): [] for L in LAYERS for fi in TOP_FEATS_PER_LAYER[L]}

for q in tqdm(char_questions, desc='characterize'):
    p = fmt(q['question'], q['options'])
    ids = tok(p, return_tensors='pt').input_ids.cuda()
    if ids.shape[1] > 1200: continue
    for L in LAYERS: captured[L] = []
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=1500, do_sample=False,
                             pad_token_id=tok.pad_token_id, use_cache=True)
    gen_ids = out[0, ids.shape[1]:].tolist()
    gen_tokens = [tok.decode([t]) for t in gen_ids]
    for L in LAYERS:
        if len(captured[L]) < 2: continue
        token_acts = torch.stack(captured[L][1:], dim=0)[:, 0, :].float()  # [T, d]
        sae = saes[L].cuda()
        with torch.no_grad():
            z, _, _ = sae.encode(token_acts.cuda())
        z_cpu = z.cpu().numpy()
        sae.cpu()
        for fi in TOP_FEATS_PER_LAYER[L]:
            for t_idx, val in enumerate(z_cpu[:, fi]):
                if val > 0 and t_idx < len(gen_tokens):
                    ctx_start = max(0, t_idx - 4)
                    ctx = ''.join(gen_tokens[ctx_start:t_idx+1])
                    feat_contexts[(L, fi)].append((float(val), gen_tokens[t_idx], ctx))

for h in handles: h.remove()

print('=== SEMANTIC CHARACTERIZATION ===')
for L in LAYERS:
    print(f'\n### L{L} top-5 discriminative features ###')
    for fi, auc, fr in feature_stats[L][:5]:
        ctxs = sorted(feat_contexts.get((L, fi), []), key=lambda x: -x[0])
        direction = 'correct↑' if auc > 0.5 else 'wrong↑'
        print(f'\nL{L} f{fi}: AUROC={auc:.3f} ({direction}), fire={fr*100:.2f}%')
        for val, tok_str, ctx in ctxs[:5]:
            ctx_clean = ctx.replace('\n', ' ')[:80]
            print(f'  z={val:.2f}  "{tok_str[:15]:<15s}"  << ...{ctx_clean}...')

with open(OUT / 'feature_characterization.json', 'w') as f:
    data = {}
    for (L, fi), ctxs in feat_contexts.items():
        if ctxs:
            data[f'L{L}_f{fi}'] = [{'z': v, 'token': t, 'context': c[:200]}
                                     for v, t, c in sorted(ctxs, key=lambda x: -x[0])[:10]]
    json.dump(data, f, indent=2, ensure_ascii=False)
api.upload_file(path_or_fileobj=str(OUT/'feature_characterization.json'),
                path_in_repo='feature_characterization.json',
                repo_id=HF_TARGET, repo_type='model',
                commit_message='top features activating tokens')

## Cell 8 — Steering eval: baseline + single-layer + combined multi-layer

In [ ]:
# Pick BEST correct-promoting feature per layer (highest AUROC > 0.5)
STEER_FEATURES = {}
for L in LAYERS:
    for fi, auc, fr in feature_stats[L]:
        if auc > 0.5 and fr > 0.005 and fr < 0.3:
            STEER_FEATURES[L] = fi
            break
print(f'Steer features: {STEER_FEATURES}')

# Eval questions (fresh, disjoint)
random.seed(777)
random.shuffle(questions)
eval_set = [q for q in questions if q not in train_questions][:20]

# Steering: install pre-hook at target layer that adds alpha * W_dec[fi] to last residual
STEER_ALPHA = 1.0
steer_config = {'layers_active': set()}  # dynamic: which layers to steer

def make_steer_hook(L, sae, fi, alpha):
    W_dec_row = sae.W_dec[fi].clone().cuda()  # [d]
    def h(mod, inp, out):
        if L in steer_config['layers_active']:
            x = out[0] if isinstance(out, tuple) else out
            delta = alpha * W_dec_row.to(x.dtype)
            x_new = x.clone()
            x_new[:, -1, :] += delta  # amplify last token
            if isinstance(out, tuple): return (x_new,) + out[1:]
            return x_new
        return out
    return h

steer_handles = []
for L in LAYERS:
    if L in STEER_FEATURES:
        sae = saes[L].cuda()
        h = get_layer_module(model, L).register_forward_hook(make_steer_hook(L, sae, STEER_FEATURES[L], STEER_ALPHA))
        steer_handles.append(h)

def gen_with_steer(q, opts, active_layers):
    p = fmt(q['question'], q['options'])
    ids = tok(p, return_tensors='pt').input_ids.cuda()
    steer_config['layers_active'] = set(active_layers)
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=MAX_NEW, do_sample=False,
                             pad_token_id=tok.pad_token_id, use_cache=True)
    return extract_answer(force_close(out[0], ids.shape[1]))

# 5 conditions: baseline (no steer) + 3 single-layer + combined
conditions = {
    'baseline': [],
    f'steer_L{LAYERS[0]}': [LAYERS[0]],
    f'steer_L{LAYERS[1]}': [LAYERS[1]],
    f'steer_L{LAYERS[2]}': [LAYERS[2]],
    'steer_combined': LAYERS,
}
results = {k: [] for k in conditions}

t0 = time.time()
for qi, q in enumerate(tqdm(eval_set, desc='steering eval')):
    gold = q['gold_letter']
    for cond_name, active in conditions.items():
        try:
            ans = gen_with_steer(q, q['options'], active)
            results[cond_name].append((ans, ans == gold))
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            results[cond_name].append((None, False))
        except Exception as e:
            results[cond_name].append((None, False))
    if (qi+1) % 3 == 0:
        print(f'  [{qi+1}/{len(eval_set)}]', end=' ')
        for k in conditions:
            acc = np.mean([r[1] for r in results[k]])
            print(f'{k}={acc:.3f}', end=' ')
        print()

for h in steer_handles: h.remove()

b = np.mean([r[1] for r in results['baseline']])
print(f'\n=== RESULTS (n={len(eval_set)}, {(time.time()-t0)/60:.0f} min) ===')
for k in conditions:
    acc = np.mean([r[1] for r in results[k]])
    delta = (acc - b) * 100 if k != 'baseline' else 0
    print(f'  {k:20s}: {acc:.3f}  Δ={delta:+.1f}pp')

## Cell 9 — Upload final results + README

In [ ]:
summary = {
    'model': MODEL_ID,
    'experiment': 'multi-layer SAE + feature steering',
    'layers': LAYERS,
    'sae_config': {'n_features': N_FEATURES, 'topk': TOPK, 'epochs': SAE_EPOCHS},
    'n_rollouts_train': len(rollouts_data),
    'n_eval': len(eval_set),
    'steer_features': {f'L{L}': int(fi) for L, fi in STEER_FEATURES.items()},
    'steer_alpha': STEER_ALPHA,
    'results': {k: float(np.mean([r[1] for r in v])) for k, v in results.items()},
    'deltas_pp': {k: float((np.mean([r[1] for r in v]) - b) * 100) for k, v in results.items() if k != 'baseline'},
}
with open(OUT / 'final_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
api.upload_file(path_or_fileobj=str(OUT/'final_summary.json'),
                path_in_repo='final_summary.json',
                repo_id=HF_TARGET, repo_type='model',
                commit_message='Final steering eval summary')
print(f'\n✅ https://huggingface.co/{HF_TARGET}')